# 04. 모델링 (MobileNetV3 + Transfer Learning)

**목표**: 반려동물 피부 질환 탐지 정확도 **90% 이상** 달성

| 항목 | 내용 |
|------|------|
| 베이스 모델 | MobileNetV3Large (ImageNet 가중치) |
| 학습 방식 | Transfer Learning → Fine-tuning 2단계 |
| 분류 대상 | A1~A7 (7클래스 피부 병변) |
| 입력 크기 | 224×224 RGB |
| 데이터 | train 24,500 / val 5,250 / test 5,250 |

### 실험 계획
| 단계 | 내용 | 목표 |
|------|------|------|
| Phase 1 | 헤드만 학습 (Frozen backbone) | 베이스라인 확보 |
| Phase 2 | 상위 레이어 Fine-tuning | 90% 달성 |


## 0. 라이브러리 로드

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('TensorFlow 버전:', tf.__version__)
print('GPU 사용 가능:', tf.config.list_physical_devices('GPU'))
print('라이브러리 로드 완료')


## 1. 경로 설정 및 하이퍼파라미터

In [ ]:
# ── 경로 설정 ──
ROOT      = Path('..')
PROCESSED = ROOT / 'data/processed'
MODEL_DIR = ROOT / 'outputs/models'
LOG_DIR   = ROOT / 'outputs/logs'
FIG_DIR   = ROOT / 'outputs/figures'

for d in [MODEL_DIR, LOG_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 하이퍼파라미터 ──
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

LESION_MAP = {
    'A1': 'A1 구진/플라크', 'A2': 'A2 비듬/각질', 'A3': 'A3 태선화/색소',
    'A4': 'A4 농포/여드름', 'A5': 'A5 미란/궤양', 'A6': 'A6 결절/종괴',
    'A7': 'A7 무증상(정상)'
}

print(f'입력 크기  : {IMG_SIZE}')
print(f'배치 크기  : {BATCH_SIZE}')


## 2. 데이터 로드 및 Generator 생성

In [ ]:
df = pd.read_csv(PROCESSED / 'dataset_cleaned.csv')

target_col = 'sub_img_path' if 'sub_img_path' in df.columns else 'img_path'

df_train = df[df['split'] == 'train'].copy()
df_val   = df[df['split'] == 'val'].copy()
df_test  = df[df['split'] == 'test'].copy()

print(f'train : {len(df_train):,}장')
print(f'val   : {len(df_val):,}장')
print(f'test  : {len(df_test):,}장')
print(f'\n클래스 분포:\n{df_train["lesion"].value_counts().sort_index()}')


In [ ]:
# ── 정규화 함수 (EDA에서 계산한 우리 데이터 mean/std) ──
MEAN = np.array([0.5616, 0.5263, 0.5056])
STD  = np.array([0.1819, 0.1841, 0.1837])

def custom_preprocess(img):
    """0~255 → EDA 채널별 z-score 정규화."""
    return (img / 255.0 - MEAN) / STD

train_datagen = ImageDataGenerator(
    preprocessing_function=custom_preprocess,
    horizontal_flip=True,
    rotation_range=30,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(
    preprocessing_function=custom_preprocess
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train, x_col=target_col, y_col='lesion',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED
)
val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=df_val, x_col=target_col, y_col='lesion',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=df_test, x_col=target_col, y_col='lesion',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

# 정규화 검증
_batch = next(iter(train_gen))
print(f'배치 픽셀 범위: min={_batch[0].min():.3f}, max={_batch[0].max():.3f}, mean={_batch[0].mean():.3f}')
print('→ 정상이면 대략 -3 ~ 3 사이, mean은 0 근처')

CLASS_INDICES = train_gen.class_indices
NUM_CLASSES   = len(CLASS_INDICES)
print(f'\n클래스 인덱스: {CLASS_INDICES}')


In [ ]:
# ── class_weight 계산 ──
lesion_classes = np.array(sorted(df_train['lesion'].unique()))
lesion_weights = compute_class_weight(
    class_weight='balanced', classes=lesion_classes, y=df_train['lesion']
)
lesion_weight_dict = {
    CLASS_INDICES[cls]: w for cls, w in zip(lesion_classes, lesion_weights)
}
print('lesion class_weight:', lesion_weight_dict)


## 3. 모델 구성 (MobileNetV3Large)

```
MobileNetV3Large (ImageNet 가중치, Frozen)
    ↓
GlobalAveragePooling2D
    ↓
BatchNormalization
    ↓
Dense(512, relu) + Dropout(0.4)
    ↓
Dense(256, relu) + Dropout(0.3)
    ↓
Dense(7, softmax)
```


In [ ]:
def build_model(num_classes: int, dropout_rate: float = 0.4):
    backbone = MobileNetV3Large(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        include_preprocessing=False
    )
    backbone.trainable = False

    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = backbone(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(dropout_rate)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(dropout_rate * 0.75)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs, name='MobileNetV3_SkinDisease'), backbone


model, backbone = build_model(NUM_CLASSES)
model.summary(show_trainable=True)


## 4. Phase 1 — Head만 학습 (Frozen Backbone)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

callbacks_phase1 = [
    ModelCheckpoint(
        MODEL_DIR / f'mobilenetv3_phase1_{timestamp}.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    CSVLogger(LOG_DIR / f'phase1_{timestamp}.csv')
]

print('Phase 1 학습 시작 — Backbone Frozen | Head 학습 중')


In [ ]:
history_p1 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    class_weight=lesion_weight_dict,
    verbose=1
)

print(f'\n[Phase 1 완료]')
print(f'  최고 val_accuracy : {max(history_p1.history["val_accuracy"]):.4f}')


## 5. 학습 곡선 시각화

In [ ]:
def plot_history(history, phase_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{phase_name} 학습 곡선', fontsize=14, fontweight='bold')
    epochs = range(1, len(history.history['loss']) + 1)

    axes[0].plot(epochs, history.history['loss'],     label='Train Loss', marker='o', markersize=3)
    axes[0].plot(epochs, history.history['val_loss'], label='Val Loss',   marker='s', markersize=3)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history.history['accuracy'],     label='Train Acc', marker='o', markersize=3)
    axes[1].plot(epochs, history.history['val_accuracy'], label='Val Acc',   marker='s', markersize=3)
    axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='목표 90%')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_history(history_p1, 'Phase 1 (Frozen Backbone)',
             save_path=FIG_DIR / f'phase1_curves_{timestamp}.png')


## 6. Phase 2 — Fine-tuning (상위 레이어 해동)

In [ ]:
UNFREEZE_FROM = -50

backbone.trainable = True
for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False

print(f'해동 레이어 수: {sum(1 for l in backbone.layers if l.trainable)}')
print(f'동결 레이어 수: {sum(1 for l in backbone.layers if not l.trainable)}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

callbacks_phase2 = [
    ModelCheckpoint(
        MODEL_DIR / f'mobilenetv3_phase2_{timestamp}.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    CSVLogger(LOG_DIR / f'phase2_{timestamp}.csv')
]

print('\nPhase 2 학습 시작 — 상위 50레이어 해동 | lr=1e-4')


In [ ]:
history_p2 = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    class_weight=lesion_weight_dict,
    verbose=1
)

best_val = max(history_p2.history['val_accuracy'])
print(f'\n[Phase 2 완료]')
print(f'  최고 val_accuracy : {best_val:.4f}')
print(f'  목표 달성 여부    : {"✅ 90% 달성!" if best_val >= 0.9 else "❌ 미달"}')


In [ ]:
plot_history(history_p2, 'Phase 2 (Fine-tuning)',
             save_path=FIG_DIR / f'phase2_curves_{timestamp}.png')


## 7. 테스트셋 최종 평가

In [ ]:
print('테스트셋 평가 중...')
test_loss, test_acc, test_top3 = model.evaluate(test_gen, verbose=1)

print(f'\n{"="*40}')
print(f'  테스트 Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Top-3 Accuracy  : {test_top3:.4f}  ({test_top3*100:.2f}%)')
print(f'  Test Loss       : {test_loss:.4f}')
print(f'  목표 달성 여부  : {"✅ 90% 달성!" if test_acc >= 0.9 else "❌ 미달"}')
print(f'{"="*40}')


In [ ]:
test_gen.reset()
y_pred_prob  = model.predict(test_gen, verbose=1)
y_pred       = np.argmax(y_pred_prob, axis=1)
y_true       = test_gen.classes
idx_to_label = {v: k for k, v in CLASS_INDICES.items()}
target_names = [LESION_MAP[idx_to_label[i]] for i in range(NUM_CLASSES)]

print('\n[클래스별 분류 리포트]')
print(classification_report(y_true, y_pred, target_names=target_names))


## 8. 혼동 행렬 시각화

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
short_names = [idx_to_label[i] for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix', fontsize=14, fontweight='bold')

sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=axes[0])
axes[0].set_title('Count'); axes[0].set_xlabel('예측'); axes[0].set_ylabel('실제')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names,
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Normalized (Recall)'); axes[1].set_xlabel('예측'); axes[1].set_ylabel('실제')

plt.tight_layout()
plt.savefig(FIG_DIR / f'confusion_matrix_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. 모델 저장

In [ ]:
final_path = MODEL_DIR / f'mobilenetv3_final_{timestamp}.keras'
model.save(final_path)
print(f'모델 저장 완료: {final_path}')

summary = {
    'timestamp'          : timestamp,
    'base_model'         : 'MobileNetV3Large',
    'normalization'      : 'EDA custom (domain-specific)',
    'phase1_best_val_acc': max(history_p1.history['val_accuracy']),
    'phase2_best_val_acc': max(history_p2.history['val_accuracy']),
    'test_acc'           : test_acc,
    'test_top3_acc'      : test_top3,
    'test_loss'          : test_loss,
    'target_achieved'    : test_acc >= 0.9
}
pd.DataFrame([summary]).to_csv(LOG_DIR / f'result_summary_{timestamp}.csv', index=False)

print()
print('=' * 50)
for k, v in summary.items():
    print(f'  {k:<28}: {v}')
print('=' * 50)


## 10. 90% 미달 시 추가 전략

| 전략 | 방법 | 예상 효과 |
|------|------|-----------|
| 더 많은 레이어 해동 | `UNFREEZE_FROM = -100` | +1~2% |
| Label Smoothing | `CategoricalCrossentropy(label_smoothing=0.1)` | 과적합 방지 |
| TTA | 추론 시 다중 증강 앙상블 | +1~2% |
| Cosine Annealing LR | LR 스케줄 변경 | 수렴 개선 |


In [ ]:
# ── [선택] Label Smoothing 적용 재학습 ──
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=5e-5),
#     loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
#     metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
# )
# history_p3 = model.fit(
#     train_gen, epochs=20, validation_data=val_gen,
#     callbacks=callbacks_phase2, class_weight=lesion_weight_dict, verbose=1
# )
print('Phase 3 코드는 주석 해제 후 실행하세요.')


---
## 노트북 완료 ✅

| Phase | 내용 | 결과 변수 |
|-------|------|----------|
| Phase 1 | Frozen Backbone + Head 학습 | `history_p1` |
| Phase 2 | 상위 50레이어 Fine-tuning | `history_p2` |
| 평가 | 테스트셋 최종 정확도 | `test_acc` |
